# Experiment 47: Full External Library Diversity + Ensemble Search

This experiment scans the external prediction library, identifies strong and structurally different prediction families, then searches a large set of rank-based ensemble recipes. Submission 14 is kept as the local baseline and is never modified.

## Plan

1. Fast-screen the full external library using a fixed sample.
2. Fully rank the strongest and most diverse finalists.
3. Build an ensemble library from the strongest candidates.
4. Search S14-anchored, external-only, diverse, and score-weighted rank blends.
5. Save candidate submissions for later Kaggle testing.

In [2]:
from pathlib import Path
import gc
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr

warnings.filterwarnings("ignore")

print("=" * 110)
print("EXP47: FULL EXTERNAL LIBRARY DIVERSITY + ENSEMBLE SEARCH")
print("=" * 110)

START_TIME = time.time()

# ============================================================================
# PATHS
# ============================================================================

ROOT = Path.cwd()

if not (ROOT / "external_data").exists():
    ROOT = ROOT.parent

BASELINE_PATH = ROOT / "submissions" / "submission_14.csv"

EXT_ROOT = ROOT / "external_data" / "s6e9_zoom_zoom_baseline"

ARCHIVE = EXT_ROOT / "ranked_predictions_latest.zip.bin"
CATALOG_PATH = EXT_ROOT / "ranked_catalog_latest.csv"

OUTPUT = ROOT / "submissions" / "experiment_47_external_ensemble"
OUTPUT.mkdir(parents=True, exist_ok=True)

SAMPLE_SIZE = 25000
RANDOM_SEED = 4701

rng = np.random.default_rng(RANDOM_SEED)

print("Project root:", ROOT)
print("Archive:", ARCHIVE)
print("Baseline:", BASELINE_PATH)

# ============================================================================
# LOAD BASELINE
# ============================================================================

baseline = pd.read_csv(BASELINE_PATH)
catalog = pd.read_csv(CATALOG_PATH)

if len(baseline) != 286571:
    raise RuntimeError(
        f"Unexpected baseline row count: {len(baseline)}"
    )

if not baseline["id"].is_unique:
    raise RuntimeError(
        "Submission 14 IDs are not unique."
    )

base_ids = baseline["id"].to_numpy()
base_pred = baseline["Will_Buy_EV"].to_numpy(dtype=np.float64)

base_rank = rankdata(
    base_pred,
    method="average"
).astype(np.float64)

print(f"Submission 14 rows: {len(baseline):,}")

# ============================================================================
# CATALOG
# ============================================================================

candidate_catalog = (
    catalog[
        catalog["public_score"].notna()
        & (catalog["rows"] == len(baseline))
    ]
    .copy()
)

print("Compatible catalog entries:", len(candidate_catalog))

if len(candidate_catalog) < 500:
    raise RuntimeError(
        "Expected the full external prediction library."
    )

# ============================================================================
# STAGE 1: FAST SCREEN
# ============================================================================

sample_idx = rng.choice(
    len(baseline),
    size=min(SAMPLE_SIZE, len(baseline)),
    replace=False
)

sample_idx.sort()

base_sample = base_pred[sample_idx]

base_sample_rank = rankdata(
    base_sample,
    method="average"
).astype(np.float64)

screen_rows = []

print("\n" + "=" * 110)
print("STAGE 1: FULL LIBRARY FAST SCREEN")
print("=" * 110)

with zipfile.ZipFile(ARCHIVE, "r") as zf:

    total = len(candidate_catalog)

    for n, row in enumerate(
        candidate_catalog.itertuples(index=False),
        start=1
    ):

        member = row.csv_path
        df = None

        try:

            with zf.open(member) as f:
                df = pd.read_csv(
                    f,
                    usecols=["id", "Will_Buy_EV"]
                )

            if len(df) != len(baseline):
                continue

            ids = df["id"].to_numpy()

            if not np.array_equal(ids, base_ids):
                continue

            pred_sample = (
                df["Will_Buy_EV"]
                .to_numpy(dtype=np.float64)[sample_idx]
            )

            candidate_rank = rankdata(
                pred_sample,
                method="average"
            ).astype(np.float64)

            rho = spearmanr(
                base_sample_rank,
                candidate_rank
            ).statistic

            rank_diff = np.abs(
                base_sample_rank - candidate_rank
            )

            public_score = float(row.public_score)

            diversity = max(
                0.0,
                1.0 - float(rho)
            )

            score_excess = max(
                0.0,
                public_score - 0.9450
            )

            screening_score = (
                score_excess * 100.0
                + diversity * 8.0
            )

            screen_rows.append({
                "csv_path": member,
                "public_score": public_score,
                "origin": row.origin,
                "author": row.author,
                "submission_id": row.submission_id,
                "version_id": row.version_id,
                "sample_spearman_to_s14": float(rho),
                "sample_mean_abs_rank_diff": float(
                    rank_diff.mean()
                ),
                "sample_median_abs_rank_diff": float(
                    np.median(rank_diff)
                ),
                "screening_score": float(
                    screening_score
                )
            })

        except Exception as e:

            print(
                "ERROR:",
                member,
                "|",
                repr(e)
            )

        finally:

            if df is not None:
                del df

        if n % 50 == 0:

            elapsed = time.time() - START_TIME

            print(
                f"Processed {n}/{total} | "
                f"elapsed {elapsed / 60:.1f} min"
            )

        gc.collect()

screen = pd.DataFrame(screen_rows)

if screen.empty:
    raise RuntimeError(
        "No compatible external predictions."
    )

print(
    "\nCompatible predictions scanned:",
    len(screen)
)

print("\nTop public scores:")

print(
    screen
    .sort_values(
        "public_score",
        ascending=False
    )[
        [
            "public_score",
            "author",
            "submission_id",
            "sample_spearman_to_s14",
            "sample_mean_abs_rank_diff"
        ]
    ]
    .head(30)
    .to_string(index=False)
)

print("\nMost diverse high-scoring candidates:")

print(
    screen[
        screen["public_score"] >= 0.9460
    ]
    .sort_values(
        [
            "sample_spearman_to_s14",
            "public_score"
        ],
        ascending=[
            True,
            False
        ]
    )[
        [
            "public_score",
            "author",
            "submission_id",
            "sample_spearman_to_s14",
            "sample_mean_abs_rank_diff",
            "screening_score"
        ]
    ]
    .head(40)
    .to_string(index=False)
)

# ============================================================================
# STAGE 2: BUILD FINALIST POOL
# ============================================================================

finalist_parts = []

# A. Highest public scores

finalist_parts.append(
    screen
    .sort_values(
        "public_score",
        ascending=False
    )
    .head(40)
)

# B. Most diverse among strong files

finalist_parts.append(
    screen[
        screen["public_score"] >= 0.9460
    ]
    .sort_values(
        [
            "sample_spearman_to_s14",
            "public_score"
        ],
        ascending=[
            True,
            False
        ]
    )
    .head(40)
)

# C. Highest screening score

finalist_parts.append(
    screen
    .sort_values(
        "screening_score",
        ascending=False
    )
    .head(40)
)

# D. Strong files in multiple score bands

for low, high in [
    (0.94650, 1.0),
    (0.94640, 0.94650),
    (0.94630, 0.94640),
    (0.94600, 0.94630)
]:

    finalist_parts.append(
        screen[
            (screen["public_score"] >= low)
            & (screen["public_score"] < high)
        ]
        .sort_values(
            "sample_spearman_to_s14"
        )
        .head(20)
    )

finalists = (
    pd.concat(finalist_parts)
    .drop_duplicates(
        subset=["csv_path"]
    )
    .copy()
)

print("\n" + "=" * 110)
print("FINALIST POOL")
print("=" * 110)

print(
    "Finalists selected:",
    len(finalists)
)

print(
    finalists
    .sort_values(
        [
            "public_score",
            "sample_spearman_to_s14"
        ],
        ascending=[
            False,
            True
        ]
    )[
        [
            "public_score",
            "author",
            "submission_id",
            "sample_spearman_to_s14",
            "screening_score"
        ]
    ]
    .head(80)
    .to_string(index=False)
)

# ============================================================================
# STAGE 2: FULL-RANK RECHECK
# ============================================================================

print("\n" + "=" * 110)
print("STAGE 2: FULL-RANK RECHECK")
print("=" * 110)

full_rows = []

with zipfile.ZipFile(ARCHIVE, "r") as zf:

    for n, row in enumerate(
        finalists.itertuples(index=False),
        start=1
    ):

        member = row.csv_path
        df = None

        try:

            with zf.open(member) as f:
                df = pd.read_csv(
                    f,
                    usecols=["id", "Will_Buy_EV"]
                )

            if not np.array_equal(
                df["id"].to_numpy(),
                base_ids
            ):
                continue

            pred = df[
                "Will_Buy_EV"
            ].to_numpy(dtype=np.float64)

            ranks = rankdata(
                pred,
                method="average"
            ).astype(np.float64)

            rho = spearmanr(
                base_rank,
                ranks
            ).statistic

            rank_diff = np.abs(
                base_rank - ranks
            )

            full_rows.append({
                "csv_path": member,
                "public_score": float(row.public_score),
                "origin": row.origin,
                "author": row.author,
                "submission_id": row.submission_id,
                "version_id": row.version_id,
                "spearman_to_s14": float(rho),
                "mean_abs_rank_diff": float(
                    rank_diff.mean()
                ),
                "median_abs_rank_diff": float(
                    np.median(rank_diff)
                ),
                "p95_abs_rank_diff": float(
                    np.percentile(
                        rank_diff,
                        95
                    )
                ),
                "p99_abs_rank_diff": float(
                    np.percentile(
                        rank_diff,
                        99
                    )
                )
            })

        except Exception as e:

            print(
                "ERROR:",
                member,
                "|",
                repr(e)
            )

        finally:

            if df is not None:
                del df

        if n % 20 == 0:

            print(
                f"Full-ranked {n}/{len(finalists)}"
            )

        gc.collect()

full = pd.DataFrame(full_rows)

print(
    "\nFully ranked finalists:",
    len(full)
)

print("\nBest score/diversity candidates:")

print(
    full[
        full["public_score"] >= 0.9460
    ]
    .sort_values(
        [
            "spearman_to_s14",
            "public_score"
        ],
        ascending=[
            True,
            False
        ]
    )[
        [
            "public_score",
            "author",
            "submission_id",
            "spearman_to_s14",
            "mean_abs_rank_diff",
            "median_abs_rank_diff",
            "p99_abs_rank_diff"
        ]
    ]
    .head(50)
    .to_string(index=False)
)

# ============================================================================
# STAGE 3: ENSEMBLE LIBRARY
# ============================================================================

selection = pd.concat([
    full
    .sort_values(
        "public_score",
        ascending=False
    )
    .head(20),

    full[
        full["public_score"] >= 0.9460
    ]
    .sort_values(
        "spearman_to_s14"
    )
    .head(20),

    full
    .sort_values(
        "mean_abs_rank_diff",
        ascending=False
    )
    .head(20),

    full
    .sort_values(
        "p99_abs_rank_diff",
        ascending=False
    )
    .head(20)
]).drop_duplicates(
    subset=["csv_path"]
)

selection = selection.head(50).copy()

print("\n" + "=" * 110)
print("ENSEMBLE LIBRARY")
print("=" * 110)

print(
    "Final ensemble candidates:",
    len(selection)
)

print(
    selection
    .sort_values(
        "public_score",
        ascending=False
    )[
        [
            "public_score",
            "author",
            "submission_id",
            "spearman_to_s14",
            "mean_abs_rank_diff"
        ]
    ]
    .to_string(index=False)
)

# ============================================================================
# LOAD FINAL RANK VECTORS
# ============================================================================

rank_vectors = {
    "s14": base_rank
}

metadata = {
    "s14": {
        "public_score": 0.94618,
        "author": "local",
        "submission_id": "submission_14"
    }
}

with zipfile.ZipFile(ARCHIVE, "r") as zf:

    for n, row in enumerate(
        selection.itertuples(index=False),
        start=1
    ):

        member = row.csv_path

        with zf.open(member) as f:

            df = pd.read_csv(
                f,
                usecols=[
                    "id",
                    "Will_Buy_EV"
                ]
            )

        pred = df[
            "Will_Buy_EV"
        ].to_numpy(dtype=np.float64)

        rank_vectors[f"ext_{n:03d}"] = rankdata(
            pred,
            method="average"
        ).astype(np.float64)

        metadata[f"ext_{n:03d}"] = {
            "public_score": float(row.public_score),
            "author": row.author,
            "submission_id": row.submission_id,
            "csv_path": member
        }

        if n % 10 == 0:

            print(
                f"Loaded full rank vectors "
                f"{n}/{len(selection)}"
            )

        del df
        gc.collect()

# ============================================================================
# HELPER FUNCTIONS
# ============================================================================

def normalize_rank(rank):

    rank_min = rank.min()
    rank_max = rank.max()

    return (
        rank - rank_min
    ) / (
        rank_max - rank_min
    )


def weighted_rank_blend(names, weights):

    result = np.zeros(
        len(base_rank),
        dtype=np.float64
    )

    total = float(
        np.sum(weights)
    )

    for name, weight in zip(
        names,
        weights
    ):

        result += (
            rank_vectors[name] * weight
        )

    return result / total


def save_candidate(name, ranks, recipe):

    out = baseline[["id"]].copy()

    out["Will_Buy_EV"] = normalize_rank(
        ranks
    )

    path = OUTPUT / f"{name}.csv"

    out.to_csv(
        path,
        index=False
    )

    corr = np.corrcoef(
        base_rank,
        ranks
    )[0, 1]

    shift = np.abs(
        base_rank - ranks
    )

    return {
        "candidate": name,
        "corr_to_s14": float(corr),
        "mean_rank_shift": float(
            shift.mean()
        ),
        "median_rank_shift": float(
            np.median(shift)
        ),
        "p99_rank_shift": float(
            np.percentile(
                shift,
                99
            )
        ),
        "recipe": recipe,
        "file": path.name
    }


# ============================================================================
# STAGE 3: ENSEMBLE SEARCH
# ============================================================================

print("\n" + "=" * 110)
print("STAGE 3: ENSEMBLE SEARCH")
print("=" * 110)

results = []

# ============================================================================
# A. S14 + INDIVIDUAL EXTERNAL FILES
# ============================================================================

individuals = [
    name
    for name in rank_vectors
    if name != "s14"
]

individuals = sorted(
    individuals,
    key=lambda x: metadata[x]["public_score"],
    reverse=True
)

for name in individuals[:30]:

    for s14_weight, ext_weight in [
        (99, 1),
        (97.5, 2.5),
        (95, 5),
        (92.5, 7.5),
        (90, 10),
        (85, 15),
        (80, 20)
    ]:

        ranks = weighted_rank_blend(
            ["s14", name],
            [s14_weight, ext_weight]
        )

        results.append(
            save_candidate(
                f"s14_ext_{name}_{s14_weight}_{ext_weight}",
                ranks,
                f"S14:{s14_weight}, {name}:{ext_weight}"
            )
        )

# ============================================================================
# B. TOP EXTERNAL PREDICTIONS, NO S14
# ============================================================================

top_external = sorted(
    individuals,
    key=lambda x: metadata[x]["public_score"],
    reverse=True
)[:20]

for k in [3, 5, 8, 12]:

    chosen = top_external[:k]

    ranks = weighted_rank_blend(
        chosen,
        [1.0] * len(chosen)
    )

    results.append(
        save_candidate(
            f"top_{k}_external_mean",
            ranks,
            "Equal rank mean of top public-score external files"
        )
    )

# ============================================================================
# C. DIVERSE EXTERNAL CONSENSUS
# ============================================================================

diverse_external = sorted(
    individuals,
    key=lambda x: (
        metadata[x]["public_score"] >= 0.9460,
        -metadata[x]["public_score"]
    ),
    reverse=True
)

selected = []

for name in diverse_external:

    if not selected:

        selected.append(name)
        continue

    current_min = min(
        np.corrcoef(
            rank_vectors[name],
            rank_vectors[x]
        )[0, 1]
        for x in selected
    )

    if current_min < 0.99998:

        selected.append(name)

    if len(selected) >= 12:
        break

print(
    "\nGreedy diverse external files selected:",
    len(selected)
)

print(selected)

for k in [4, 6, 8, 10, 12]:

    chosen = selected[:k]

    if len(chosen) < k:
        continue

    ranks = weighted_rank_blend(
        chosen,
        [1.0] * len(chosen)
    )

    results.append(
        save_candidate(
            f"diverse_{k}_external_mean",
            ranks,
            "Equal rank mean of greedily diversified external files"
        )
    )

# ============================================================================
# D. S14 + DIVERSE CONSENSUS
# ============================================================================

for ext_weight in [
    2.5,
    5,
    7.5,
    10,
    15,
    20,
    25,
    30
]:

    chosen = selected[:8]

    names = ["s14"] + chosen

    weights = (
        [100 - ext_weight]
        + [ext_weight / len(chosen)] * len(chosen)
    )

    ranks = weighted_rank_blend(
        names,
        weights
    )

    results.append(
        save_candidate(
            f"s14_diverse8_ext_{ext_weight}",
            ranks,
            f"S14:{100-ext_weight}, diverse8:{ext_weight}"
        )
    )

# ============================================================================
# E. SCORE-WEIGHTED EXTERNAL CONSENSUS
# ============================================================================

score_candidates = [
    name
    for name in individuals
    if metadata[name]["public_score"] >= 0.9460
]

score_candidates = sorted(
    score_candidates,
    key=lambda x: metadata[x]["public_score"],
    reverse=True
)[:25]

for temperature in [
    0.25,
    0.5,
    1.0,
    2.0
]:

    raw_weights = np.array([
        np.exp(
            (
                metadata[name]["public_score"]
                - 0.9460
            ) / temperature
        )
        for name in score_candidates
    ])

    ranks = weighted_rank_blend(
        score_candidates,
        raw_weights
    )

    results.append(
        save_candidate(
            f"score_weighted_t{temperature}",
            ranks,
            f"Public-score exponential weighting, temperature={temperature}"
        )
    )

# ============================================================================
# F. S14 + SCORE-WEIGHTED CONSENSUS
# ============================================================================

for ext_weight in [
    5,
    10,
    15,
    20,
    25
]:

    raw_weights = np.array([
        np.exp(
            (
                metadata[name]["public_score"]
                - 0.9460
            ) / 0.5
        )
        for name in score_candidates
    ])

    names = ["s14"] + score_candidates

    weights = np.concatenate([
        [100 - ext_weight],
        raw_weights / raw_weights.sum() * ext_weight
    ])

    ranks = weighted_rank_blend(
        names,
        weights
    )

    results.append(
        save_candidate(
            f"s14_score_weighted_{ext_weight}",
            ranks,
            f"S14:{100-ext_weight}, score-weighted external:{ext_weight}"
        )
    )

# ============================================================================
# G. BEST EXTERNAL ANCHOR + DIVERSE CORRECTION
# ============================================================================

best_external = top_external[0]

for correction_weight in [
    1,
    2.5,
    5,
    7.5,
    10,
    15,
    20
]:

    chosen = selected[:8]

    correction_names = [
        x
        for x in chosen
        if x != best_external
    ]

    names = [
        best_external
    ] + correction_names

    weights = (
        [100 - correction_weight]
        + [
            correction_weight / len(correction_names)
        ] * len(correction_names)
    )

    ranks = weighted_rank_blend(
        names,
        weights
    )

    results.append(
        save_candidate(
            f"best_ext_diverse_correction_{correction_weight}",
            ranks,
            f"Best external:{100-correction_weight}, diverse correction:{correction_weight}"
        )
    )

# ============================================================================
# STAGE 4: FIND MOST PROMISING CANDIDATES
# ============================================================================

result_df = pd.DataFrame(results)

result_df = (
    result_df
    .drop_duplicates(
        subset=["candidate"]
    )
)

print("\n" + "=" * 110)
print("EXP47 CANDIDATE SUMMARY")
print("=" * 110)

print(
    result_df
    .sort_values(
        "mean_rank_shift",
        ascending=False
    )[
        [
            "candidate",
            "corr_to_s14",
            "mean_rank_shift",
            "median_rank_shift",
            "p99_rank_shift",
            "recipe"
        ]
    ]
    .head(80)
    .to_string(index=False)
)

print("\n" + "=" * 110)
print("SMALLEST RANK PERTURBATIONS")
print("=" * 110)

print(
    result_df
    .sort_values(
        "corr_to_s14",
        ascending=False
    )[
        [
            "candidate",
            "corr_to_s14",
            "mean_rank_shift",
            "median_rank_shift",
            "recipe"
        ]
    ]
    .head(25)
    .to_string(index=False)
)

# ============================================================================
# SAVE ANALYSIS
# ============================================================================

screen.to_csv(
    OUTPUT / "exp47_full_library_screen.csv",
    index=False
)

full.to_csv(
    OUTPUT / "exp47_finalist_full_recheck.csv",
    index=False
)

selection.to_csv(
    OUTPUT / "exp47_ensemble_library.csv",
    index=False
)

result_df.to_csv(
    OUTPUT / "exp47_candidate_summary.csv",
    index=False
)

print("\nSaved analysis files:")

print(
    OUTPUT / "exp47_full_library_screen.csv"
)

print(
    OUTPUT / "exp47_finalist_full_recheck.csv"
)

print(
    OUTPUT / "exp47_ensemble_library.csv"
)

print(
    OUTPUT / "exp47_candidate_summary.csv"
)

print("\n" + "=" * 110)
print("EXP47 COMPLETE")
print("=" * 110)

elapsed = time.time() - START_TIME

print(
    f"Total runtime: {elapsed / 60:.1f} minutes"
)

print(
    f"Candidate CSVs created: {len(result_df)}"
)

EXP47: FULL EXTERNAL LIBRARY DIVERSITY + ENSEMBLE SEARCH
Project root: c:\Users\aakif\Documents\DataCompetition
Archive: c:\Users\aakif\Documents\DataCompetition\external_data\s6e9_zoom_zoom_baseline\ranked_predictions_latest.zip.bin
Baseline: c:\Users\aakif\Documents\DataCompetition\submissions\submission_14.csv
Submission 14 rows: 286,571
Compatible catalog entries: 649

STAGE 1: FULL LIBRARY FAST SCREEN
Processed 50/649 | elapsed 0.4 min
Processed 100/649 | elapsed 0.7 min
Processed 150/649 | elapsed 1.1 min
Processed 200/649 | elapsed 1.5 min
Processed 250/649 | elapsed 1.9 min
Processed 300/649 | elapsed 2.2 min
Processed 350/649 | elapsed 2.6 min
Processed 400/649 | elapsed 2.9 min
Processed 450/649 | elapsed 3.3 min
Processed 500/649 | elapsed 3.7 min
Processed 550/649 | elapsed 4.0 min
Processed 600/649 | elapsed 4.4 min

Compatible predictions scanned: 649

Top public scores:
 public_score   author  submission_id  sample_spearman_to_s14  sample_mean_abs_rank_diff
      0.94657